# Demo D2. The wave equation and Fourier series

**Partial differential equations · vibrating string.**

A string fixed at both ends obeys $u_{tt} = c^2 u_{xx}$ on $x \in [0, \pi]$ with $u(0,t) = u(\pi,t) = 0$. Separation of variables gives standing-wave modes $\sin(nx)$, each oscillating in time at its own frequency $\omega_n = nc$. Any motion is a superposition of these modes: the initial shape $u(x,0)=\varphi(x)$ and initial velocity $u_t(x,0)=\psi(x)$ fix how much of each mode is present. This lab builds that Fourier-series solution and animates it with a time slider.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The solution

On $[0,\pi]$ with fixed ends the solution is
$$ u(x,t) = \sum_{n=1}^{\infty} \big[a_n \cos(n c\,t) + b_n \sin(n c\,t)\big]\sin(nx), $$
where the coefficients come from the sine series of the initial data,
$$ a_n = \frac{2}{\pi}\int_0^\pi \varphi(x)\sin(nx)\,dx, \qquad b_n = \frac{2}{n c \pi}\int_0^\pi \psi(x)\sin(nx)\,dx. $$
Each mode keeps its shape and simply oscillates; the string's motion is the sum. We take $c=1$ and truncate the series at $N$ terms.

In [ ]:
# ---------------------------------------------------------------------------
# Domain, coefficient integrals, and the initial-data catalog. Each scenario is
# a pair (phi, psi): the initial displacement u(x,0) and velocity u_t(x,0).
# ---------------------------------------------------------------------------
c = 1.0                               # wave speed
L = np.pi                             # domain [0, L]
xf = np.linspace(0.0, L, 400)         # fine grid for the coefficient integrals

def sine_coeffs(g, N):
    """Sine-series coefficients (2/L) integral_0^L g(x) sin(n x) dx, n=1..N,
    by the trapezoidal rule on the uniform grid xf."""
    n = np.arange(1, N + 1)
    integrand = g(xf)[:, None] * np.sin(np.outer(xf, n))
    h = xf[1] - xf[0]
    integral = (integrand.sum(axis=0) - 0.5 * (integrand[0] + integrand[-1])) * h
    return (2.0 / L) * integral

def _triangle(x):
    return np.where(x <= L / 2, x, L - x)

def _impulse(x):                      # narrow Gaussian velocity ~ delta(x - pi/4)
    sigma = 0.15
    return np.exp(-((x - L / 4) ** 2) / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

SCENARIOS = {
    "standing wave  sin(x)":        (lambda x, m: np.sin(x),                 lambda x, m: np.zeros_like(x)),
    "harmonic  sin(m x)":           (lambda x, m: np.sin(m * x),             lambda x, m: np.zeros_like(x)),
    "mixed position + velocity":    (lambda x, m: 4 * np.sin(x),             lambda x, m: np.sin(2 * x)),
    "sum of sines":                 (lambda x, m: (np.sin(x) + np.sin(2*x)
                                                   + np.sin(3*x)/2 + np.sin(4*x)/6),
                                                                             lambda x, m: np.zeros_like(x)),
    "plucked string (triangle)":    (lambda x, m: _triangle(x),              lambda x, m: np.zeros_like(x)),
    "hammer strike (impulse)":      (lambda x, m: np.zeros_like(x),          lambda x, m: _impulse(x)),
}

In [ ]:
# ---------------------------------------------------------------------------
# Assemble the coefficients for a scenario and evaluate the truncated series.
# ---------------------------------------------------------------------------
def coefficients(scenario, m, N):
    phi, psi = SCENARIOS[scenario]
    n = np.arange(1, N + 1)
    a = sine_coeffs(lambda x: phi(x, m), N)
    b = sine_coeffs(lambda x: psi(x, m), N) / (n * c)     # divide by omega_n = n c
    return a, b

def field(x, t, a, b):
    """u(x, t) = sum_n [a_n cos(n c t) + b_n sin(n c t)] sin(n x)."""
    n = np.arange(1, len(a) + 1)
    temporal = a * np.cos(n * c * t) + b * np.sin(n * c * t)
    spatial = np.sin(np.outer(x, n))
    return spatial @ temporal

## Watch it move

Pick an initial condition and drag the **time** slider. `harmonic` uses the integer $m$; `terms` is how many Fourier modes are summed (raise it to sharpen a corner or a spike).

In [ ]:
def show_wave(scenario="standing wave  sin(x)", m=3, terms=40, t=0.0):
    a, b = coefficients(scenario, m, terms)
    x = np.linspace(0.0, L, 400)

    # A stable y-limit: the largest displacement seen over one period.
    amp = max(np.abs(field(x, tau, a, b)).max()
              for tau in np.linspace(0.0, 2 * np.pi / c, 25))
    amp = max(amp, 1e-6)

    plt.figure()
    plt.plot(x, field(x, t, a, b), "b-", lw=2.5)
    plt.fill_between(x, field(x, t, a, b), color="#2563eb", alpha=0.08)
    plt.axhline(0.0, color="0.7", lw=1)
    plt.scatter([0, L], [0, 0], color="k", zorder=5)
    plt.xlim(0, L); plt.ylim(-1.15 * amp, 1.15 * amp)
    plt.xlabel("x"); plt.ylabel("u(x, t)")
    plt.title(f"Vibrating string   (t = {t:.2f})")
    plt.show()

interact(
    show_wave,
    scenario=Dropdown(options=list(SCENARIOS), value="standing wave  sin(x)", description="initial"),
    m=IntSlider(value=3, min=1, max=8, step=1, description="harmonic m"),
    terms=IntSlider(value=40, min=1, max=60, step=1, description="terms N"),
    t=FloatSlider(value=0.0, min=0.0, max=2 * np.pi, step=0.05, description="time t"),
);

## Which modes are present?

The energy in mode $n$ is set by its amplitude $\sqrt{a_n^2 + b_n^2}$. A pure harmonic has a single nonzero bar; a plucked or struck string spreads its energy across many modes, with the sharp features carried by the high-$n$ tail.

In [ ]:
def show_spectrum(scenario="plucked string (triangle)", m=3, terms=40):
    a, b = coefficients(scenario, m, terms)
    n = np.arange(1, terms + 1)
    plt.figure()
    plt.bar(n, np.sqrt(a**2 + b**2), color="#2563eb")
    plt.xlabel("mode n"); plt.ylabel(r"amplitude  $\sqrt{a_n^2 + b_n^2}$")
    plt.title("Fourier mode amplitudes")
    plt.show()

interact(
    show_spectrum,
    scenario=Dropdown(options=list(SCENARIOS), value="plucked string (triangle)", description="initial"),
    m=IntSlider(value=3, min=1, max=8, step=1, description="harmonic m"),
    terms=IntSlider(value=40, min=1, max=60, step=1, description="terms N"),
);

## Things to try

- A single `harmonic  sin(m x)` stays a pure standing wave: it oscillates in place with $m-1$ interior nodes and never changes shape.
- The `plucked` and `hammer` scenarios need many terms; drop `terms` to $2$ or $3$ and watch the corner or spike smooth over (Gibbs behavior).
- `mixed position + velocity` combines a $\cos$ mode and a $\sin$ mode at different frequencies, so the shape genuinely travels.

## Summary

- The fixed-end wave equation solves as a sum of standing modes $\sin(nx)$, each oscillating at $\omega_n = nc$.
- Initial displacement sets the $\cos(\omega_n t)$ (the $a_n$); initial velocity sets the $\sin(\omega_n t)$ (the $b_n$).
- Sharp initial features live in the high-$n$ coefficients, so they need many terms to resolve.